In [1]:
import pandas as pd
import re
from pathlib import Path

# 1) Load file
csv_path = Path("cleaned_cleanser1_update.csv")
if not csv_path.exists():
    csv_path = next(Path(".").glob("**/cleaned_cleanser1_update.csv"), None)
if csv_path is None:
    raise FileNotFoundError("File cleaned_cleanser1_update.csv tidak ditemukan di folder notebook Anda.")

df = pd.read_csv(csv_path)
print(f"Jumlah baris dataset: {len(df)}")
print(f"Kolom yang tersedia: {list(df.columns)}")

# 2) Fungsi bantu untuk memilih kolom yang sesuai
def pick_col(columns, aliases):
    cols = [str(c).strip().lower() for c in columns]
    for alias in aliases:
        alias_l = alias.strip().lower()
        for c, cl in zip(columns, cols):
            if cl == alias_l:
                return c
        for c, cl in zip(columns, cols):
            if alias_l in cl.replace(" ", "_"):
                return c
    return None

# 3) Normalisasi teks
def norm(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("/", " ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 4) Cek apakah value ada di alamat / text
def value_in_text(text, value):
    t = norm(text)
    v = norm(value)
    if not v:
        return False
    if v in t:
        return True
    
    # variasi umum:
    for prefix in ["kabupaten ", "kota ", "kecamatan ", "desa ", "kelurahan ", "provinsi "]:
        if v.startswith(prefix):
            v2 = v.replace(prefix, "", 1)
            if v2 in t:
                return True
    return False

# 5) Deteksi kolom utama
name_cols = ["nama_sppg", "nama sppg", "nama_sppgnya", "sppg_name", "name", "nama"]
prov_cols = ["provinsi_sppg", "propinsi_sppg", "propvinsi_sppg", "provinsi", "province", "provinsi_sppgnya"]
kec_cols = ["kecamatan_sppg", "kecamatan", "kecamatan_sppgnya", "district"]
kab_cols = ["kabupaten_kota_sppg", "kabupaten_kota", "kabupaten", "kota", "kabupaten_kota_sppgnya", "city"]
addr_cols = ["alamat_sppg", "alamat_sppgnya", "alamat_lengkap", "alamat", "address"]

name_col = pick_col(df.columns, name_cols)
prov_col = pick_col(df.columns, prov_cols)
kec_col = pick_col(df.columns, kec_cols)
kab_col = pick_col(df.columns, kab_cols)
addr_col = pick_col(df.columns, addr_cols)

if name_col is None or prov_col is None or kec_col is None or kab_col is None or addr_col is None:
    missing = []
    for c, label in [(name_col, "nama sppg"), (prov_col, "provinsi sppg"), (kec_col, "kecamatan sppg"), (kab_col, "kabupaten/kota sppg"), (addr_col, "alamat sppg")]:
        if c is None:
            missing.append(label)
    raise ValueError(f"Kolom yang dibutuhkan tidak ditemukan. Kolom yang kurang: {missing}. Silakan cek nama kolom di CSV.")

# 6) Siapkan data untuk pengecekan
out = df[[name_col, prov_col, kec_col, kab_col, addr_col]].copy()
out.columns = ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg"]

# 7) Cek kecocokan tiap data terhadap alamat
out["prov_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["provinsi_sppg"]), axis=1)
out["kec_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["kecamatan_sppg"]), axis=1)
out["kab_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["kabupaten_kota_sppg"]), axis=1)
out["semua_sesuai"] = out["prov_match"] & out["kec_match"] & out["kab_match"]

# 8) Ringkasan
jumlah_data = len(out)
jumlah_sesuai = int(out["semua_sesuai"].sum())
jumlah_tidak_sesuai = jumlah_data - jumlah_sesuai

print(f"Jumlah nama SPPG yang dicek: {jumlah_data}")
print(f"Jumlah data sesuai semua (provinsi + kecamatan + kab/kota): {jumlah_sesuai}")
print(f"Jumlah data tidak sama / berbeda: {jumlah_tidak_sesuai}")

# 9) Tampilkan data yang tidak sesuai
mismatch = out.loc[~out["semua_sesuai"], [
    "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg",
    "prov_match", "kec_match", "kab_match"
]].copy()

print("\nData yang tidak sesuai / berbeda:")
display(mismatch)

# Opsional: simpan hasil ke CSV
out.to_csv("hasil_identifikasi_sppg.csv", index=False)
print("\nHasil pengecekan sudah disimpan ke: hasil_identifikasi_sppg.csv")

Jumlah baris dataset: 6241
Kolom yang tersedia: ['No SPPG', 'Nama SPPG', 'Provinsi SPPG', 'Kabupaten/Kota SPPG', 'Kecamatan SPPG', 'Latitude', 'Longitude', 'Alamat SPPG', 'link', 'Koordinat Valid']
Jumlah nama SPPG yang dicek: 6241
Jumlah data sesuai semua (provinsi + kecamatan + kab/kota): 3980
Jumlah data tidak sama / berbeda: 2261

Data yang tidak sesuai / berbeda:


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
4,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
20,SPPG ACEH UTARA T JAMBOE AYE TANJONG ARA 2,Aceh,T. Jambo Aye,Aceh Utara,3FW8+JG3 SPPG ACEH UTARA T JAMBOE AYE TANJONG ...,True,False,True
21,SPPG KECAMATAN JEUMPA,Aceh,Jeumpa,Aceh Barat Daya,"Jl. Medan - Banda Aceh, Blang Bladeh, Kec. Jeu...",True,True,False
30,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
32,SPPG PIDIE INDRAJAYA BLANG GAROT,Aceh,Indra Jaya,Aceh Jaya,"8WRG+86G SPPG PIDIE INDRAJAYA BLANG GAROT, Bla...",True,True,False
...,...,...,...,...,...,...,...,...
6220,SPPG TOLITOLI UTARA - YAYASAN DHAMMA METTA NUS...,Sulawesi Tengah,Toli-Toli Utara,Toli Toli,8XPJ+P3G SPPG TOLITOLI UTARA - YAYASAN DHAMMA ...,True,True,False
6221,SPPG SIBEA,Sulawesi Tengah,Lampasio,Toli Toli,"WV36+4FC SPPG SIBEA, Sibea, Kec. Lampasio, Kab...",True,True,False
6222,SPPG DONDO MALULU,Sulawesi Tengah,Dondo,Toli Toli,"QG98+JWF SPPG DONDO MALULU, Malulu, Kec. Dondo...",True,True,False
6223,SPPG KARINA LAMPASIO,Sulawesi Tengah,Lampasio,Toli Toli,"Lampasio, Kec. Lampasio, Kabupaten Toli-Toli, ...",True,True,False



Hasil pengecekan sudah disimpan ke: hasil_identifikasi_sppg.csv


In [2]:
# 1) Buka data hasil deteksi yang tidak sesuai
hasil_path = Path("hasil_identifikasi_sppg.csv")
if not hasil_path.exists():
    hasil_path = next(Path(".").glob("**/hasil_identifikasi_sppg.csv"), None)
if hasil_path is None:
    raise FileNotFoundError("File hasil_identifikasi_sppg.csv tidak ditemukan.")

hasil = pd.read_csv(hasil_path)

# Ambil baris yang masih tidak sesuai
mismatch_rows = hasil.loc[~hasil["semua_sesuai"]].copy()
print(f"Jumlah data yang perlu diperbaiki: {len(mismatch_rows)}")

# 2) Helper normalisasi teks
def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("/", " ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 3) Ambil kandidat unik untuk tiap kolom dari df yang valid
def unique_values(series):
    vals = series.dropna().astype(str).str.strip()
    vals = vals[vals != ""]
    return sorted(set(vals), key=lambda x: len(x), reverse=True)

prov_vals = unique_values(df["Provinsi SPPG"])
kab_vals = unique_values(df["Kabupaten/Kota SPPG"])
kec_vals = unique_values(df["Kecamatan SPPG"])

# 4) Cari nilai terbaik yang ada di alamat
def best_value_from_address(addr, candidates):
    addr_n = norm_text(addr)
    if not addr_n:
        return None

    found = []
    for c in candidates:
        c_n = norm_text(c)
        if not c_n:
            continue
        if c_n in addr_n:
            found.append((len(c_n), c))
    if not found:
        return None
    return max(found, key=lambda x: x[0])[1]

# 5) Cari index row di df berdasarkan Nama SPPG + alamat
def find_df_index_by_name(row):
    name_key = str(row["nama_sppg"]).strip().lower()
    candidates = df.index[df["Nama SPPG"].astype(str).str.strip().str.lower() == name_key].tolist()
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        return None

    addr_key = norm_text(row.get("alamat_sppg", ""))
    for idx in candidates:
        if norm_text(df.at[idx, "Alamat SPPG"]) == addr_key:
            return idx
    return candidates[0]

# 6) Perbaiki nilai di df berdasarkan alamat
fixed_count = 0

for _, r in mismatch_rows.iterrows():
    idx = find_df_index_by_name(r)
    if idx is None:
        continue

    addr = df.at[idx, "Alamat SPPG"]

    # Provinsi
    if pd.notna(r.get("prov_match")) and r["prov_match"] is False:
        new_prov = best_value_from_address(addr, prov_vals)
        if new_prov is not None:
            df.at[idx, "Provinsi SPPG"] = new_prov
            fixed_count += 1

    # Kabupaten/Kota
    if pd.notna(r.get("kab_match")) and r["kab_match"] is False:
        new_kab = best_value_from_address(addr, kab_vals)
        if new_kab is not None:
            df.at[idx, "Kabupaten/Kota SPPG"] = new_kab
            fixed_count += 1

    # Kecamatan
    if pd.notna(r.get("kec_match")) and r["kec_match"] is False:
        new_kec = best_value_from_address(addr, kec_vals)
        if new_kec is not None:
            df.at[idx, "Kecamatan SPPG"] = new_kec
            fixed_count += 1

print(f"Jumlah baris yang berhasil diperbaiki: {fixed_count}")

# 7) Validasi ulang hasil perbaikan
out_fixed = df[["Nama SPPG", "Provinsi SPPG", "Kecamatan SPPG", "Kabupaten/Kota SPPG", "Alamat SPPG"]].copy()
out_fixed.columns = ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg"]

out_fixed["prov_match"] = out_fixed.apply(lambda r: value_in_text(r["alamat_sppg"], r["provinsi_sppg"]), axis=1)
out_fixed["kec_match"] = out_fixed.apply(lambda r: value_in_text(r["alamat_sppg"], r["kecamatan_sppg"]), axis=1)
out_fixed["kab_match"] = out_fixed.apply(lambda r: value_in_text(r["alamat_sppg"], r["kabupaten_kota_sppg"]), axis=1)
out_fixed["semua_sesuai"] = out_fixed["prov_match"] & out_fixed["kec_match"] & out_fixed["kab_match"]

jumlah_data = len(out_fixed)
jumlah_sesuai = int(out_fixed["semua_sesuai"].sum())
jumlah_tidak_sesuai = jumlah_data - jumlah_sesuai

print(f"Jumlah total data: {jumlah_data}")
print(f"Jumlah data sesuai setelah perbaikan: {jumlah_sesuai}")
print(f"Jumlah data tidak sesuai setelah perbaikan: {jumlah_tidak_sesuai}")

# 8) Tampilkan hasil yang masih tidak sesuai
mismatch_fixed = out_fixed.loc[~out_fixed["semua_sesuai"], [
    "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg",
    "prov_match", "kec_match", "kab_match"
]].copy()

print("\nData yang masih tidak sesuai setelah perbaikan:")
display(mismatch_fixed)

# 9) Simpan data hasil perbaikan ke file baru
df.to_csv("cleaned_cleanser1_update_fixed.csv", index=False)
out_fixed.to_csv("hasil_identifikasi_sppg_fixed.csv", index=False)

print("\nFile hasil perbaikan sudah disimpan:")
print("- cleaned_cleanser1_update_fixed.csv")
print("- hasil_identifikasi_sppg_fixed.csv")

Jumlah data yang perlu diperbaiki: 2261
Jumlah baris yang berhasil diperbaiki: 1997
Jumlah total data: 6241
Jumlah data sesuai setelah perbaikan: 5097
Jumlah data tidak sesuai setelah perbaikan: 1144

Data yang masih tidak sesuai setelah perbaikan:


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
4,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
20,SPPG ACEH UTARA T JAMBOE AYE TANJONG ARA 2,Aceh,T. Jambo Aye,Aceh Utara,3FW8+JG3 SPPG ACEH UTARA T JAMBOE AYE TANJONG ...,True,False,True
30,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
145,SPPG TANGSI LAMA 2,Aceh,Seruway,Aceh Tamiang,"85WG+R7Q SPPG Tangsi lama 2, Tangsi Lama, Kec....",True,False,True
190,SPPG KRUENG SEUNONG,Aceh,Kuta Makmur,Aceh Utara,"Jl. Elak No.KM. 8, Krueng Seunong, Kuta Makmur...",True,True,False
...,...,...,...,...,...,...,...,...
6219,DAPUR GIZI SPPG MFL GALANG,Sulawesi Tengah,Galang,Toli Toli,"Jalan bandar udara, Ginunggung, Kec. Galang, K...",True,True,False
6220,SPPG TOLITOLI UTARA - YAYASAN DHAMMA METTA NUS...,Sulawesi Tengah,Toli-Toli Utara,Toli Toli,8XPJ+P3G SPPG TOLITOLI UTARA - YAYASAN DHAMMA ...,True,True,False
6221,SPPG SIBEA,Sulawesi Tengah,Lampasio,Toli Toli,"WV36+4FC SPPG SIBEA, Sibea, Kec. Lampasio, Kab...",True,True,False
6222,SPPG DONDO MALULU,Sulawesi Tengah,Dondo,Toli Toli,"QG98+JWF SPPG DONDO MALULU, Malulu, Kec. Dondo...",True,True,False



File hasil perbaikan sudah disimpan:
- cleaned_cleanser1_update_fixed.csv
- hasil_identifikasi_sppg_fixed.csv


In [3]:
# 1) Buka file hasil perbaikan terakhir
df = pd.read_csv("cleaned_cleanser1_update_fixed.csv")
hasil = pd.read_csv("hasil_identifikasi_sppg_fixed.csv")

mismatch = hasil.loc[~hasil["semua_sesuai"]].copy()
print(f"Data yang masih tidak sesuai: {len(mismatch)}")

# 2) Helper normalisasi cepat
def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-").replace("/", " ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def make_sorted_candidates(series):
    vals = []
    seen = set()
    for v in series.dropna().astype(str):
        v = v.strip()
        if not v:
            continue
        nv = norm_text(v)
        if nv and nv not in seen:
            seen.add(nv)
            vals.append((nv, v))
    return sorted(vals, key=lambda x: len(x[0]), reverse=True)

prov_cands = make_sorted_candidates(df["Provinsi SPPG"])
kab_cands = make_sorted_candidates(df["Kabupaten/Kota SPPG"])
kec_cands = make_sorted_candidates(df["Kecamatan SPPG"])

def best_from_address(addr, cand_list):
    addr_n = norm_text(addr)
    if not addr_n:
        return None
    best_v = None
    best_len = -1
    for norm_v, value in cand_list:
        if norm_v and norm_v in addr_n and len(norm_v) > best_len:
            best_v = value
            best_len = len(norm_v)
    return best_v

# 3) Peta nama -> indeks (cepat, tanpa iterrows)
name_to_idx = {}
for i, name in df["Nama SPPG"].items():
    k = norm_text(name)
    if k:
        name_to_idx.setdefault(k, []).append(i)

def find_df_index(name, addr):
    key = norm_text(name)
    cand = name_to_idx.get(key, [])
    if not cand:
        return None
    if len(cand) == 1:
        return cand[0]
    addr_key = norm_text(addr)
    for i in cand:
        if norm_text(df.at[i, "Alamat SPPG"]) == addr_key:
            return i
    return cand[0]

# 4) Perbaiki sisa data yang tidak sesuai
fixed_count = 0

for row in mismatch.itertuples(index=False):
    idx = find_df_index(getattr(row, "nama_sppg"), getattr(row, "alamat_sppg"))
    if idx is None:
        continue

    addr = df.at[idx, "Alamat SPPG"]

    if not row.prov_match:
        new_prov = best_from_address(addr, prov_cands)
        if new_prov is not None and str(new_prov) != str(df.at[idx, "Provinsi SPPG"]):
            df.at[idx, "Provinsi SPPG"] = new_prov
            fixed_count += 1

    if not row.kec_match:
        new_kec = best_from_address(addr, kec_cands)
        if new_kec is not None and str(new_kec) != str(df.at[idx, "Kecamatan SPPG"]):
            df.at[idx, "Kecamatan SPPG"] = new_kec
            fixed_count += 1

    if not row.kab_match:
        new_kab = best_from_address(addr, kab_cands)
        if new_kab is not None and str(new_kab) != str(df.at[idx, "Kabupaten/Kota SPPG"]):
            df.at[idx, "Kabupaten/Kota SPPG"] = new_kab
            fixed_count += 1

print(f"Jumlah baris yang berhasil diperbaiki: {fixed_count}")

# 5) Validasi ulang semua data
out = df[["Nama SPPG", "Provinsi SPPG", "Kecamatan SPPG", "Kabupaten/Kota SPPG", "Alamat SPPG"]].copy()
out.columns = ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg"]

out["prov_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["provinsi_sppg"]), axis=1)
out["kec_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["kecamatan_sppg"]), axis=1)
out["kab_match"] = out.apply(lambda r: value_in_text(r["alamat_sppg"], r["kabupaten_kota_sppg"]), axis=1)
out["semua_sesuai"] = out["prov_match"] & out["kec_match"] & out["kab_match"]

total = len(out)
ok = int(out["semua_sesuai"].sum())
bad = total - ok

print(f"Total data: {total}")
print(f"Data sesuai setelah perbaikan: {ok}")
print(f"Data masih tidak sesuai: {bad}")

# 6) Tampilkan hasil yang masih gagal
if bad > 0:
    remain = out.loc[~out["semua_sesuai"], [
        "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg",
        "prov_match", "kec_match", "kab_match"
    ]].copy()
    print("\nData yang masih tersisa tidak sesuai:")
    display(remain)

# 7) Simpan hasil final
df.to_csv("cleaned_cleanser1_update_fixed.csv", index=False)
out.to_csv("hasil_identifikasi_sppg_fixed.csv", index=False)

print("\nFile final tersimpan:")
print("- cleaned_cleanser1_update_fixed.csv")
print("- hasil_identifikasi_sppg_fixed.csv")

Data yang masih tidak sesuai: 1144
Jumlah baris yang berhasil diperbaiki: 0
Total data: 6241
Data sesuai setelah perbaikan: 5097
Data masih tidak sesuai: 1144

Data yang masih tersisa tidak sesuai:


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
4,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
20,SPPG ACEH UTARA T JAMBOE AYE TANJONG ARA 2,Aceh,T. Jambo Aye,Aceh Utara,3FW8+JG3 SPPG ACEH UTARA T JAMBOE AYE TANJONG ...,True,False,True
30,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
145,SPPG TANGSI LAMA 2,Aceh,Seruway,Aceh Tamiang,"85WG+R7Q SPPG Tangsi lama 2, Tangsi Lama, Kec....",True,False,True
190,SPPG KRUENG SEUNONG,Aceh,Kuta Makmur,Aceh Utara,"Jl. Elak No.KM. 8, Krueng Seunong, Kuta Makmur...",True,True,False
...,...,...,...,...,...,...,...,...
6219,DAPUR GIZI SPPG MFL GALANG,Sulawesi Tengah,Galang,Toli Toli,"Jalan bandar udara, Ginunggung, Kec. Galang, K...",True,True,False
6220,SPPG TOLITOLI UTARA - YAYASAN DHAMMA METTA NUS...,Sulawesi Tengah,Toli-Toli Utara,Toli Toli,8XPJ+P3G SPPG TOLITOLI UTARA - YAYASAN DHAMMA ...,True,True,False
6221,SPPG SIBEA,Sulawesi Tengah,Lampasio,Toli Toli,"WV36+4FC SPPG SIBEA, Sibea, Kec. Lampasio, Kab...",True,True,False
6222,SPPG DONDO MALULU,Sulawesi Tengah,Dondo,Toli Toli,"QG98+JWF SPPG DONDO MALULU, Malulu, Kec. Dondo...",True,True,False



File final tersimpan:
- cleaned_cleanser1_update_fixed.csv
- hasil_identifikasi_sppg_fixed.csv


In [4]:
# 1) Pastikan data yang masih gagal benar-benar dibetulkan berdasarkan nama SPPG + alamat SPPG
#    lalu sesuaikan provinsi, kecamatan, kabupaten/kota sesuai string yang muncul di alamat
if "mismatch" not in globals():
    mismatch = hasil.loc[~hasil["semua_sesuai"]].copy()

# helper normalisasi
def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-").replace("/", " ").replace("_", " ")
    s = s.replace("kabupaten", "kabupaten ").replace("kecamatan", "kecamatan ").replace("provinsi", "provinsi ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"[\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_from_address(addr, cand_list):
    if pd.isna(addr):
        return None
    addr_n = norm_text(addr)
    if not addr_n:
        return None
    best_val = None
    best_len = -1
    for norm_v, raw_v in cand_list:
        if norm_v and norm_v in addr_n and len(norm_v) > best_len:
            best_val = raw_v
            best_len = len(norm_v)
    return best_val

# parser alamat yang lebih kuat
def extract_addr_part(addr, labels):
    if pd.isna(addr):
        return None
    s = str(addr)
    for label in labels:
        m = re.search(rf"(?:{label})\s+([A-Za-z0-9][A-Za-z0-9 .\-]*)", s, flags=re.I)
        if m:
            val = re.sub(r"\s+", " ", m.group(1)).strip()
            if val:
                return val
    return None

# cari index row sesuai nama SPPG + alamat SPPG
name_to_idx = {}
for i, name in df["Nama SPPG"].items():
    k = norm_text(name)
    if k:
        name_to_idx.setdefault(k, []).append(i)

def find_df_index(name_val, addr_val):
    key = norm_text(name_val)
    cand = name_to_idx.get(key, [])
    if not cand:
        return None
    if len(cand) == 1:
        return cand[0]
    addr_key = norm_text(addr_val)
    for i in cand:
        if norm_text(df.at[i, "Alamat SPPG"]) == addr_key:
            return i
    return cand[0]

fixed_count = 0

for row in mismatch.itertuples(index=False):
    idx = find_df_index(getattr(row, "nama_sppg"), getattr(row, "alamat_sppg"))
    if idx is None:
        continue

    addr = df.at[idx, "Alamat SPPG"]
    if pd.isna(addr):
        continue

    # 1) Provinsi
    if row.prov_match is False:
        new_val = (
            extract_addr_part(addr, ["provinsi", "prov\\.?", "prov"])
            or best_from_address(addr, prov_cands)
        )
        if new_val is not None and str(new_val).strip() and str(new_val) != str(df.at[idx, "Provinsi SPPG"]):
            df.at[idx, "Provinsi SPPG"] = new_val
            fixed_count += 1

    # 2) Kecamatan
    if row.kec_match is False:
        new_val = (
            extract_addr_part(addr, ["kecamatan", "kec\\.?", "kec"])
            or best_from_address(addr, kec_cands)
        )
        if new_val is not None and str(new_val).strip() and str(new_val) != str(df.at[idx, "Kecamatan SPPG"]):
            df.at[idx, "Kecamatan SPPG"] = new_val
            fixed_count += 1

    # 3) Kabupaten/Kota
    if row.kab_match is False:
        new_val = (
            extract_addr_part(addr, ["kabupaten", "kab\\.?", "kab", "kota"])
            or best_from_address(addr, kab_cands)
        )
        if new_val is not None and str(new_val).strip() and str(new_val) != str(df.at[idx, "Kabupaten/Kota SPPG"]):
            df.at[idx, "Kabupaten/Kota SPPG"] = new_val
            fixed_count += 1

print(f"Jumlah baris berhasil diperbaiki: {fixed_count}")

# 2) Validasi ulang semua data
out = df[["Nama SPPG", "Provinsi SPPG", "Kecamatan SPPG", "Kabupaten/Kota SPPG", "Alamat SPPG"]].copy()
out.columns = ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg"]

# gunakan fungsi match yang lebih kuat: hilangkan tanda hubung/pemisah untuk membandingkan
def match_value(addr, value):
    if pd.isna(addr) or pd.isna(value):
        return False
    return norm_text(value) in norm_text(addr)

out["prov_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["provinsi_sppg"]), axis=1)
out["kec_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["kecamatan_sppg"]), axis=1)
out["kab_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["kabupaten_kota_sppg"]), axis=1)
out["semua_sesuai"] = out["prov_match"] & out["kec_match"] & out["kab_match"]

total = len(out)
ok = int(out["semua_sesuai"].sum())
bad = total - ok

print(f"Total data: {total}")
print(f"Data sesuai setelah perbaikan: {ok}")
print(f"Data masih tidak sesuai: {bad}")

if bad > 0:
    remain = out.loc[~out["semua_sesuai"], [
        "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg",
        "prov_match", "kec_match", "kab_match"
    ]].copy()
    print("\nData yang masih tidak sesuai setelah perbaikan:")
    display(remain)

# 3) Simpan hasil final
df.to_csv("cleaned_cleanser1_update_fixed.csv", index=False)
out.to_csv("hasil_identifikasi_sppg_fixed.csv", index=False)

print("\nFile final tersimpan:")
print("- cleaned_cleanser1_update_fixed.csv")
print("- hasil_identifikasi_sppg_fixed.csv")

Jumlah baris berhasil diperbaiki: 122
Total data: 6241
Data sesuai setelah perbaikan: 5088
Data masih tidak sesuai: 1153

Data yang masih tidak sesuai setelah perbaikan:


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
4,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
30,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
74,SPPG CAHAYA PEMBANGUNAN NUSANTARA,Aceh,Banda Raya,Kota Banda Aceh,"G8F6+CR2 SPPG Cahaya Pembangunan Nusantara, Jl...",True,True,False
78,SPPG BANDA ACEH KUTA ALAM MULIA 2,Aceh,Kuta Alam,Kota Banda Aceh,"Jl. Teungku Di Blang No.Kel, Mulia, Kuta Alam,...",True,True,False
80,SPPG BAITURRAHMAN SUKARAMAI,Aceh,Baiturrahman,Kota Banda Aceh,"Sukaramai, Baiturrahman, Banda Aceh City, Aceh...",True,True,False
...,...,...,...,...,...,...,...,...
6200,KARYA KASIH (SPPG TOMADO LINDU),Sulawesi Tengah,Lindu,Sigi,"Tomado, Kec. Lindu, Palu, Sulawesi Tengah 94363",True,True,False
6201,SPPG SIGI GUMBASA PAKULI,Sulawesi Tengah,Gumbasa,Sigi,"QW6V+63G SPPG Sigi Gumbasa Pakuli, Jl. Poros P...",False,True,True
6202,SPPG BABAKAN CIPARAY 2 KOTA BANDUNG,Bandung,Babakan Ciparay,Kota Bandung,"Jl. Caringin No.35, Babakan Ciparay, Bandung C...",True,True,False
6208,SPPG LOLU SIGI BIROMARU,Sulawesi Tengah,Sigi Biromaru,Sigi,"2WQ8+WGJ SPPG LOLU SIGI BIROMARU, Jl. Lasoso, ...",False,True,True



File final tersimpan:
- cleaned_cleanser1_update_fixed.csv
- hasil_identifikasi_sppg_fixed.csv


In [5]:
# Tampilkan jumlah data yang masih belum sesuai
print(f"Jumlah data yang masih belum sesuai: {len(remain)}")

# Tampilkan seluruh tabelnya secara keseluruhan
display(
    remain[
        ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg", "prov_match", "kec_match", "kab_match"]
    ].reset_index(drop=True)
)

Jumlah data yang masih belum sesuai: 1153


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
0,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
1,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
2,SPPG CAHAYA PEMBANGUNAN NUSANTARA,Aceh,Banda Raya,Kota Banda Aceh,"G8F6+CR2 SPPG Cahaya Pembangunan Nusantara, Jl...",True,True,False
3,SPPG BANDA ACEH KUTA ALAM MULIA 2,Aceh,Kuta Alam,Kota Banda Aceh,"Jl. Teungku Di Blang No.Kel, Mulia, Kuta Alam,...",True,True,False
4,SPPG BAITURRAHMAN SUKARAMAI,Aceh,Baiturrahman,Kota Banda Aceh,"Sukaramai, Baiturrahman, Banda Aceh City, Aceh...",True,True,False
...,...,...,...,...,...,...,...,...
1148,KARYA KASIH (SPPG TOMADO LINDU),Sulawesi Tengah,Lindu,Sigi,"Tomado, Kec. Lindu, Palu, Sulawesi Tengah 94363",True,True,False
1149,SPPG SIGI GUMBASA PAKULI,Sulawesi Tengah,Gumbasa,Sigi,"QW6V+63G SPPG Sigi Gumbasa Pakuli, Jl. Poros P...",False,True,True
1150,SPPG BABAKAN CIPARAY 2 KOTA BANDUNG,Bandung,Babakan Ciparay,Kota Bandung,"Jl. Caringin No.35, Babakan Ciparay, Bandung C...",True,True,False
1151,SPPG LOLU SIGI BIROMARU,Sulawesi Tengah,Sigi Biromaru,Sigi,"2WQ8+WGJ SPPG LOLU SIGI BIROMARU, Jl. Lasoso, ...",False,True,True


In [6]:
# Cell 7: perbaiki sisa 1153 data yang masih belum sesuai berdasarkan teks alamat SPPG
# Tujuan: selaraskan "Kecamatan SPPG", "Kabupaten/Kota SPPG", dan "Provinsi SPPG" mengikuti data yang ada di "Alamat SPPG"

# 1) bantu normalisasi
def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-").replace("/", " ").replace("_", " ")
    s = s.replace("kabupaten", "kabupaten ").replace("kecamatan", "kecamatan ")
    s = s.replace("provinsi", "provinsi ").replace("kota", "kota ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"[\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_from_address(addr, cand_list):
    if pd.isna(addr):
        return None
    addr_n = norm_text(addr)
    if not addr_n:
        return None
    best_val = None
    best_len = -1
    for norm_v, raw_v in cand_list:
        if norm_v and norm_v in addr_n and len(norm_v) > best_len:
            best_val = raw_v
            best_len = len(norm_v)
    return best_val

def extract_label_value(addr, labels):
    if pd.isna(addr):
        return None
    s = str(addr)
    # urutan prioritas label
    patterns = [
        rf"\b(?:{lab})\.?\s+([A-Za-z0-9][A-Za-z0-9 .\-]*)" for lab in labels
    ]
    for pat in patterns:
        m = re.search(pat, s, flags=re.I)
        if m:
            val = re.sub(r"\s+", " ", m.group(1)).strip(" ,.;-")
            if val and val.lower() not in {"kecamatan", "kabupaten", "kota", "city", "provinsi", "propinsi"}:
                return val
    return None

# 2) cari indeks row dari nama + alamat
if "name_to_idx" not in globals():
    name_to_idx = {}
    for i, name in df["Nama SPPG"].items():
        k = norm_text(name)
        if k:
            name_to_idx.setdefault(k, []).append(i)

def find_df_index_by_name(name_val, addr_val):
    key = norm_text(name_val)
    cand = name_to_idx.get(key, [])
    if not cand:
        return None
    if len(cand) == 1:
        return cand[0]
    addr_key = norm_text(addr_val)
    for i in cand:
        if norm_text(df.at[i, "Alamat SPPG"]) == addr_key:
            return i
    return cand[0]

# 3) perbaiki baris yang masih salah
fixed_count = 0

for _, r in remain.iterrows():
    idx = find_df_index_by_name(r["nama_sppg"], r["alamat_sppg"])
    if idx is None:
        continue

    addr = df.at[idx, "Alamat SPPG"]
    if pd.isna(addr):
        continue

    # Provinsi
    if r["prov_match"] is False:
        new_prov = (
            extract_label_value(addr, ["provinsi", "propinsi", "prov"])
            or best_from_address(addr, prov_cands)
        )
        if new_prov is not None and str(new_prov).strip() and str(new_prov) != str(df.at[idx, "Provinsi SPPG"]):
            df.at[idx, "Provinsi SPPG"] = new_prov
            fixed_count += 1

    # Kecamatan
    if r["kec_match"] is False:
        new_kec = (
            extract_label_value(addr, ["kecamatan", "kec"])
            or best_from_address(addr, kec_cands)
        )
        if new_kec is not None and str(new_kec).strip() and str(new_kec) != str(df.at[idx, "Kecamatan SPPG"]):
            df.at[idx, "Kecamatan SPPG"] = new_kec
            fixed_count += 1

    # Kabupaten / Kota
    if r["kab_match"] is False:
        new_kab = (
            extract_label_value(addr, ["kabupaten", "kab", "kota", "city"])
            or best_from_address(addr, kab_cands)
        )
        if new_kab is not None and str(new_kab).strip() and str(new_kab) != str(df.at[idx, "Kabupaten/Kota SPPG"]):
            df.at[idx, "Kabupaten/Kota SPPG"] = new_kab
            fixed_count += 1

print(f"Jumlah baris berhasil diperbaiki pada langkah ini: {fixed_count}")

# 4) validasi ulang semua data
out = df[["Nama SPPG", "Provinsi SPPG", "Kecamatan SPPG", "Kabupaten/Kota SPPG", "Alamat SPPG"]].copy()
out.columns = ["nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg"]

def match_value(addr, value):
    if pd.isna(addr) or pd.isna(value):
        return False
    return norm_text(value) in norm_text(addr)

out["prov_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["provinsi_sppg"]), axis=1)
out["kec_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["kecamatan_sppg"]), axis=1)
out["kab_match"] = out.apply(lambda r: match_value(r["alamat_sppg"], r["kabupaten_kota_sppg"]), axis=1)
out["semua_sesuai"] = out["prov_match"] & out["kec_match"] & out["kab_match"]

total = len(out)
ok = int(out["semua_sesuai"].sum())
bad = total - ok

print(f"Total data: {total}")
print(f"Data sesuai: {ok}")
print(f"Data masih belum sesuai: {bad}")

remain = out.loc[~out["semua_sesuai"], [
    "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg", "alamat_sppg",
    "prov_match", "kec_match", "kab_match"
]].copy()

if bad > 0:
    print("\nData yang masih belum sesuai setelah perbaikan:")
    display(remain.reset_index(drop=True))
else:
    print("\nSemua data sudah sesuai setelah perbaikan.")
    
# 5) simpan hasil final
df.to_csv("cleaned_cleanser1_update_fixed.csv", index=False)
out.to_csv("hasil_identifikasi_sppg_fixed.csv", index=False)

print("\nFile final tersimpan:")
print("- cleaned_cleanser1_update_fixed.csv")
print("- hasil_identifikasi_sppg_fixed.csv")

Jumlah baris berhasil diperbaiki pada langkah ini: 78
Total data: 6241
Data sesuai: 5145
Data masih belum sesuai: 1096

Data yang masih belum sesuai setelah perbaikan:


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match
0,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Meureubo,Aceh Barat,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,True,False
1,SPPG KUTELINTANG YAYASAN TUNAS BANGSA AGARA,Aceh,Blangkejeren,Gayo Lues,"Jalan lintas Blangkejeren-Takengon, Kuta Linta...",True,True,False
2,SPPG CAHAYA PEMBANGUNAN NUSANTARA,Aceh,Banda Raya,Kota Banda Aceh,"G8F6+CR2 SPPG Cahaya Pembangunan Nusantara, Jl...",True,True,False
3,SPPG BANDA ACEH KUTA ALAM MULIA 2,Aceh,Kuta Alam,Kota Banda Aceh,"Jl. Teungku Di Blang No.Kel, Mulia, Kuta Alam,...",True,True,False
4,SPPG BAITURRAHMAN SUKARAMAI,Aceh,Baiturrahman,Kota Banda Aceh,"Sukaramai, Baiturrahman, Banda Aceh City, Aceh...",True,True,False
...,...,...,...,...,...,...,...,...
1091,SPPG PARANGGI,NaN,NaN,NaN,NaN,False,False,False
1092,KARYA KASIH (SPPG TOMADO LINDU),Sulawesi Tengah,Lindu,Sigi,"Tomado, Kec. Lindu, Palu, Sulawesi Tengah 94363",True,True,False
1093,SPPG SIGI GUMBASA PAKULI,Sulawesi Tengah,Gumbasa,Sigi,"QW6V+63G SPPG Sigi Gumbasa Pakuli, Jl. Poros P...",False,True,True
1094,SPPG LOLU SIGI BIROMARU,Sulawesi Tengah,Sigi Biromaru,Sigi,"2WQ8+WGJ SPPG LOLU SIGI BIROMARU, Jl. Lasoso, ...",False,True,True



File final tersimpan:
- cleaned_cleanser1_update_fixed.csv
- hasil_identifikasi_sppg_fixed.csv


In [8]:
# Data yang sudah sesuai (semua_sesuai == True) disimpan ke CSV
if "out" in globals():
    data_ok = out.loc[out["semua_sesuai"]].copy()
elif "hasil" in globals():
    data_ok = hasil.loc[hasil["semua_sesuai"]].copy()
else:
    raise NameError("Variabel 'out' atau 'hasil' tidak ditemukan.")

# Pastikan kolom yang akan disimpan konsisten
cols = [
    "nama_sppg", "provinsi_sppg", "kecamatan_sppg", "kabupaten_kota_sppg",
    "alamat_sppg", "prov_match", "kec_match", "kab_match", "semua_sesuai"
]
data_ok = data_ok.reindex(columns=cols).reset_index(drop=True)

csv_name_ok = "data_sudah_sesuai.csv"
data_ok.to_csv(csv_name_ok, index=False)

print(f"Jumlah data yang sudah sesuai: {len(data_ok)}")
print(f"File CSV sudah dibuat: {csv_name_ok}")
display(data_ok.head())

Jumlah data yang sudah sesuai: 5145
File CSV sudah dibuat: data_sudah_sesuai.csv


,nama_sppg,provinsi_sppg,kecamatan_sppg,kabupaten_kota_sppg,alamat_sppg,prov_match,kec_match,kab_match,semua_sesuai
0,SPPG WOYLA TIMUR PASI JANENG,Aceh,Woyla Timur,Aceh Barat,"C3X3+X6P SPPG Woyla Timur Pasi Janeng, Pasi Ja...",True,True,True,True
1,SPPG WOYLA KUALA BHEE,Aceh,Woyla,Aceh Barat,"92MV+RCR SPPG WOYLA KUALA BHEE, Kuala Bhee, Ke...",True,True,True,True
2,SPPG ACEH BARAT SAMATIGA SUAK TIMAH,Aceh,Samatiga,Aceh Barat,"Jl. Meulaboh - Banda Aceh, Suak Timah, Kec. Sa...",True,True,True,True
3,SPPG PANTE CEUREUMEN,Aceh,Pante Ceureumen,Aceh Barat,"C77V+XR4 SPPG PANTE CEUREUMEN, Pante Ceureumen...",True,True,True,True
4,SPPG KAWAY XVI,Aceh,Kaway XVI,Aceh Barat,"6565+46R SPPG KAWAY XVI, Alue Tampak, Kec. Kaw...",True,True,True,True


In [7]:
# Simpan data yang masih belum rapi / belum sesuai ke CSV
if "remain" not in globals():
    remain = out.loc[~out["semua_sesuai"]].copy()

csv_name = "data_belum_rapih_dan_sama.csv"

remain_csv = remain[
    [
        "nama_sppg",
        "provinsi_sppg",
        "kecamatan_sppg",
        "kabupaten_kota_sppg",
        "alamat_sppg",
        "prov_match",
        "kec_match",
        "kab_match",
    ]
].reset_index(drop=True)

remain_csv.to_csv(csv_name, index=False)

print(f"Jumlah data yang disimpan: {len(remain_csv)}")
print(f"File CSV sudah dibuat: {csv_name}")
display(remain_csv.head())

PermissionError: [Errno 13] Permission denied: 'data_belum_rapih_dan_sama.csv'

In [1]:
from pathlib import Path
import pandas as pd
import re

def find_file(filename):
    p = Path(filename)
    if p.exists():
        return p
    found = next(Path(".").glob(f"**/{filename}"), None)
    return found

def pick_col(columns, aliases):
    cols = [str(c).strip().lower() for c in columns]
    for alias in aliases:
        a = str(alias).strip().lower()
        for c, cl in zip(columns, cols):
            if cl == a:
                return c
        for c, cl in zip(columns, cols):
            if a in cl.replace(" ", "_"):
                return c
    return None

def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-").replace("/", " ").replace("_", " ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"[\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Baca source CSV utama
src_path = find_file("cleaned_cleanser1_update.csv")
if src_path is None:
    raise FileNotFoundError("File cleaned_cleanser1_update.csv tidak ditemukan.")
src = pd.read_csv(src_path)

src_name_col = pick_col(src.columns, ["nama_sppg", "nama sppg", "name", "nama"])
src_addr_col = pick_col(src.columns, ["alamat_sppg", "alamat sppg", "alamat_lengkap", "alamat", "address"])
src_lat_col = pick_col(src.columns, ["latitude", "lat", "latitude_sppg", "latitute", "latitude sppg"])
src_lon_col = pick_col(src.columns, ["longitude", "lon", "long", "longitude_sppg", "longitude sppg"])

if src_name_col is None or src_addr_col is None or src_lat_col is None or src_lon_col is None:
    raise ValueError(
        f"Kolom latitude/longitude/nama/alamat tidak ditemukan di {src_path}. "
        f"Kolom yang ada: {list(src.columns)}"
    )

src = src[[src_name_col, src_addr_col, src_lat_col, src_lon_col]].copy()
src.columns = ["nama_sppg", "alamat_sppg", "latitude", "longitude"]

# Normalisasi nama dan alamat untuk matching
src["nama_sppg_norm"] = src["nama_sppg"].map(norm_text)
src["alamat_sppg_norm"] = src["alamat_sppg"].map(norm_text)

# Hapus baris kosong
src = src.dropna(subset=["nama_sppg"], how="all").copy()

# 2) Helper untuk mencari row dari source
def find_source_row(name_val, addr_val):
    name_key = norm_text(name_val)
    addr_key = norm_text(addr_val)

    # 1) exact name + exact address
    if name_key:
        cand = src[(src["nama_sppg_norm"] == name_key)]
        if not cand.empty:
            if addr_key:
                exact = cand[cand["alamat_sppg_norm"] == addr_key]
                if not exact.empty:
                    return exact.iloc[0]
            return cand.iloc[0]

    # 2) fallback by name only
    if name_key:
        cand2 = src[src["nama_sppg_norm"] == name_key]
        if not cand2.empty:
            return cand2.iloc[0]

    # 3) fallback by address only (jarang dipakai)
    if addr_key:
        cand3 = src[src["alamat_sppg_norm"] == addr_key]
        if not cand3.empty:
            return cand3.iloc[0]

    return None

# 3) Proses file target yang diminta
target_files = ["data_belum_rapih_dan_sama.csv", "data_sudah_sesuai.csv"]

for target_name in target_files:
    target_path = find_file(target_name)
    if target_path is None:
        print(f"File {target_name} tidak ditemukan, dilewati.")
        continue

    target = pd.read_csv(target_path)

    # Pastikan kolom nama/alamat ada
    t_name_col = pick_col(target.columns, ["nama_sppg", "nama sppg", "name", "nama"])
    t_addr_col = pick_col(target.columns, ["alamat_sppg", "alamat sppg", "alamat_lengkap", "alamat", "address"])

    if t_name_col is None or t_addr_col is None:
        print(f"File {target_name} tidak punya kolom nama/alamat yang kompatibel, dilewati.")
        continue

    # Tambahkan kolom longitude/latitude jika belum ada
    if "latitude" not in target.columns:
        target["latitude"] = pd.NA
    if "longitude" not in target.columns:
        target["longitude"] = pd.NA

    matched = 0
    not_matched = 0

    for idx, row in target.iterrows():
        name_val = row.get(t_name_col, None)
        addr_val = row.get(t_addr_col, None)

        src_match = find_source_row(name_val, addr_val)
        if src_match is not None:
            lat = src_match["latitude"]
            lon = src_match["longitude"]
            target.at[idx, "latitude"] = lat
            target.at[idx, "longitude"] = lon
            matched += 1
        else:
            not_matched += 1

    # Simpan hasil ke file yang sama
    target.to_csv(target_path, index=False)

    print(f"File: {target_name}")
    print(f"  Matching latitude/longitude berhasil: {matched}")
    print(f"  Tidak ditemukan match: {not_matched}")
    print(f"  Disimpan ke: {target_path}")

print("\nProses penambahan longitude dan latitude selesai.")

File: data_belum_rapih_dan_sama.csv
  Matching latitude/longitude berhasil: 1096
  Tidak ditemukan match: 0
  Disimpan ke: data_belum_rapih_dan_sama.csv
File: data_sudah_sesuai.csv
  Matching latitude/longitude berhasil: 5145
  Tidak ditemukan match: 0
  Disimpan ke: data_sudah_sesuai.csv

Proses penambahan longitude dan latitude selesai.


In [3]:
from pathlib import Path
import pandas as pd
import re

def find_file(filename):
    p = Path(filename)
    if p.exists():
        return p
    found = next(Path(".").glob(f"**/{filename}"), None)
    return found

def pick_col(columns, aliases):
    cols = [str(c).strip().lower() for c in columns]
    for alias in aliases:
        a = str(alias).strip().lower()
        for c, cl in zip(columns, cols):
            if cl == a:
                return c
        for c, cl in zip(columns, cols):
            if a in cl.replace(" ", "_"):
                return c
    return None

def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = s.replace("–", "-").replace("—", "-").replace("/", " ").replace("_", " ")
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"[\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Baca source utama
src_path = find_file("cleaned_cleanser1_update.csv")
if src_path is None:
    raise FileNotFoundError("File cleaned_cleanser1_update.csv tidak ditemukan.")

src = pd.read_csv(src_path)

src_name_col = pick_col(src.columns, ["nama_sppg", "nama sppg", "name", "nama"])
if src_name_col is None:
    raise ValueError(f"Kolom nama SPPG tidak ditemukan di {src_path}. Kolom yang ada: {list(src.columns)}")

# Cari kolom link di source
link_aliases = ["link", "links", "url", "website", "sppg_link", "link_sppg", "link_url", "maps_link", "google_maps_link"]
src_link_col = None
for alias in link_aliases:
    col = pick_col(src.columns, [alias])
    if col is not None:
        src_link_col = col
        break

if src_link_col is None:
    raise ValueError(f"Kolom link tidak ditemukan di {src_path}. Kolom yang ada: {list(src.columns)}")

# 2) Siapkan mapping nama SPPG -> link dari source
src2 = src[[src_name_col, src_link_col]].copy()
src2.columns = ["nama_sppg", "link"]

src2["nama_sppg_norm"] = src2["nama_sppg"].map(norm_text)
src2 = src2.dropna(subset=["nama_sppg"], how="all").copy()
src2 = src2.dropna(subset=["link"]).copy()
src2 = src2.drop_duplicates(subset=["nama_sppg_norm"], keep="last").reset_index(drop=True)

name_to_link = dict(zip(src2["nama_sppg_norm"], src2["link"]))

# 3) Proses file target
for target_name in ["data_sudah_sesuai.csv", "data_belum_rapih_dan_sama.csv"]:
    target_path = find_file(target_name)
    if target_path is None:
        print(f"File {target_name} tidak ditemukan, dilewati.")
        continue

    target = pd.read_csv(target_path)

    target_name_col = pick_col(target.columns, ["nama_sppg", "nama sppg", "name", "nama"])
    if target_name_col is None:
        print(f"File {target_name} tidak punya kolom nama SPPG, dilewati.")
        continue

    if "link" not in target.columns:
        target["link"] = None

    matched = 0
    not_found = 0

    for idx, row in target.iterrows():
        name_key = norm_text(row.get(target_name_col, ""))
        link_val = name_to_link.get(name_key)
        if pd.notna(link_val) and str(link_val).strip():
            target.at[idx, "link"] = link_val
            matched += 1
        else:
            not_found += 1

    target.to_csv(target_path, index=False)
    print(f"{target_name}: link matched = {matched}, tidak ditemukan = {not_found}")

print("\nProses penambahan kolom link dari cleaned_cleanser1_update.csv selesai.")

data_sudah_sesuai.csv: link matched = 5145, tidak ditemukan = 0
data_belum_rapih_dan_sama.csv: link matched = 1096, tidak ditemukan = 0

Proses penambahan kolom link dari cleaned_cleanser1_update.csv selesai.
